# Layer Normalization — Layer Normalization (2016)

_Papers · Foundations & Optimization_

**Normalize each training case across its own features (not across the batch), so the layer behaves identically at any batch size — even batch size 1.**

---

This notebook is a practice scaffold for the **Layer Normalization — Layer Normalization (2016)** lesson. We build LayerNorm from scratch, prove it matches PyTorch, and watch it stay identical at any batch size. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — PyTorch

### Step 1 — Define LayerNorm from scratch

LayerNorm normalizes each example using **its own features**, not the batch. For a row `x` of `H` features we compute the mean and (biased) variance *across those features* (`dim=-1`), standardize to mean 0 / std ~1, then apply a learned per-feature gain `gamma` and bias `beta`. Because every statistic comes from the single example in front of it, the output never depends on what else is in the batch.

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)

def my_layernorm(x, gamma, beta, eps=1e-5):
    """LayerNorm from scratch — Eq. 3 + transform (Eq. 5) of Ba, Kiros & Hinton (2016).
       x: (batch, features). Statistics are taken ACROSS THE FEATURES (dim -1),
       one (mu, sigma) per example — NOT down the batch like BatchNorm."""
    mu  = x.mean(dim=-1, keepdim=True)                  # mean over the H features (Eq. 3)
    var = x.var(dim=-1, unbiased=False, keepdim=True)   # biased variance, /H (Eq. 3)
    xhat = (x - mu) / torch.sqrt(var + eps)             # normalize -> mean 0, std ~1
    return gamma * xhat + beta                          # learned gain & bias (Eq. 5)


### Step 2 — Check it against `nn.LayerNorm`

The surest test of a from-scratch implementation is to match the library it reimplements. We build PyTorch's `nn.LayerNorm`, give its gain and bias non-trivial random values (so a sloppy implementation can't pass by accident), and confirm our output is identical on a batch of 8.

In [ ]:
# ---- THE ORACLE: my LN must equal nn.LayerNorm ----
H = 5
x = torch.randn(8, H)

ref = nn.LayerNorm(H)                    # default eps=1e-5, elementwise gain=1, bias=0
with torch.no_grad():                    # give g, b non-trivial values to make the test real
    ref.weight.normal_()
    ref.bias.normal_()

mine = my_layernorm(x, ref.weight, ref.bias)
ok = torch.allclose(mine, ref(x), atol=1e-6)
print("allclose vs nn.LayerNorm (batch 8):", ok)        # expect True


### Step 3 — Show batch-size independence

This is the whole point of LayerNorm. We take the *same* first example, run it alone (batch of 1), and confirm two things: our output still matches `nn.LayerNorm`, and that example's result is byte-for-byte the same whether it sat in a batch of 8 or stood alone. BatchNorm would fail the last check, because its statistics are pooled across the batch.

In [ ]:
# ---- batch-size INDEPENDENCE: same example, batch of 1 ----
one = x[:1]                              # a single example, shape (1, H)

ok1 = torch.allclose(my_layernorm(one, ref.weight, ref.bias),
                     ref(one), atol=1e-6)
print("allclose vs nn.LayerNorm (batch 1):", ok1)       # expect True

same = torch.allclose(my_layernorm(one, ref.weight, ref.bias)[0],
                      mine[0], atol=1e-6)
print("same example's output is identical at batch 1 vs batch 8:", same)  # expect True
# (BatchNorm would FAIL this last line — its output for an example depends on the batch.)


### Step 4 — Reproduce the worked example and drop it into a net

Finally we sanity-check against the lesson's hand-worked example (features `[2, 4, 4, 6]`, gain 2, bias 1) and then place a real `nn.LayerNorm` inside a small 2-layer network, between the linear layer and the activation, to confirm it composes cleanly.

In [ ]:
# ---- recompute the worked example: features [2,4,4,6], g=2, b=1 ----
xe = torch.tensor([[2., 4., 4., 6.]])
g  = torch.full((4,), 2.0)
b  = torch.full((4,), 1.0)
print("worked example y:", my_layernorm(xe, g, b).squeeze().tolist())  # ~ [-1.828, 1.0, 1.0, 3.828]

# ---- drop it into a 2-layer net (LN goes between linear and activation) ----
net = nn.Sequential(nn.Linear(H, 16), nn.LayerNorm(16), nn.ReLU(), nn.Linear(16, 2))
print("net output shape:", net(torch.randn(3, H)).shape)  # torch.Size([3, 2])


## Visualize the data & results

_Feed the SAME example through normalization once inside a batch of 8 and once alone (batch of 1). LayerNorm uses only that example's own features — does its output stay identical, where BatchNorm's would shift?_

### Step 1 — Set up one fixed example inside a batch

We make a batch of 8 random examples but pin example 0 to a known vector `[1, 3, 5, 7]`. We'll normalize this same example two ways — inside the full batch and alone — under both LayerNorm and BatchNorm, and compare.

In [ ]:
import numpy as np

rng = np.random.default_rng(0)

H = 4
# a batch of 8 examples; example index 0 is our fixed [1,3,5,7]
batch = rng.normal(0, 3, (8, H))
batch[0] = [1., 3., 5., 7.]
eps = 1e-5


### Step 2 — LayerNorm: normalize across features

LayerNorm works along `axis=1` (the features), giving each row its own mean and variance. We normalize example 0 once inside the batch of 8 and once on its own, and confirm the two outputs are identical — the rest of the batch is irrelevant.

In [ ]:
# --- LayerNorm: normalize ACROSS FEATURES (axis=1), per example ---
def layernorm(x):
    mu = x.mean(axis=1, keepdims=True)
    var = x.var(axis=1, keepdims=True)          # biased, /H
    return (x - mu) / np.sqrt(var + eps)

ln_in_batch = layernorm(batch)[0]               # example 0 inside the batch of 8
ln_alone    = layernorm(batch[:1])[0]           # example 0 normalized alone (batch 1)
print("LayerNorm in batch:", np.round(ln_in_batch, 4))
print("LayerNorm alone   :", np.round(ln_alone, 4))
print("identical?", np.allclose(ln_in_batch, ln_alone))   # True


### Step 3 — BatchNorm: normalize down the batch

BatchNorm instead works along `axis=0` (down the batch), so each feature's statistics are pooled across all examples. Run the same comparison: example 0's output now *changes* when it's alone — in fact a batch of 1 has zero variance, so every value collapses to 0. This is exactly the batch dependence LayerNorm avoids.

In [ ]:
# --- BatchNorm: normalize DOWN THE BATCH (axis=0), per feature ---
def batchnorm(x):
    mu = x.mean(axis=0, keepdims=True)
    var = x.var(axis=0, keepdims=True)          # biased, /N
    return (x - mu) / np.sqrt(var + eps)

bn_in_batch = batchnorm(batch)[0]               # example 0 inside the batch of 8
bn_alone    = batchnorm(batch[:1])[0]           # batch of 1 -> variance 0 -> all zeros
print("BatchNorm in batch:", np.round(bn_in_batch, 4))
print("BatchNorm alone   :", np.round(bn_alone, 4))         # ~[0,0,0,0]
print("identical?", np.allclose(bn_in_batch, bn_alone))    # False


## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** LayerNorm the single feature vector [1, 3, 5, 7] (H=4), then apply gain $g=2$, bias $b=0$. (Ignore $\epsilon$.)

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Mean over the 4 features: $\mu=(1+3+5+7)/4=4$. — _Average across this example's own features._
- Variance: $\sigma^2=((1-4)^2+(3-4)^2+(5-4)^2+(7-4)^2)/4=(9+1+1+9)/4=5$; $\sigma=\sqrt5\approx2.236$. — _Spread over the features; biased (divide by $H=4$)._
- Normalize: $[(1-4)/2.236,\,(3-4)/2.236,\,(5-4)/2.236,\,(7-4)/2.236]=[-1.342,-0.447,0.447,1.342]$. — _Mean 0, standard deviation ~1._
- Scale/shift: $2\cdot[-1.342,-0.447,0.447,1.342]+0=[-2.683,-0.894,0.894,2.683]$. — _Learned gain stretches; bias 0 leaves the center at 0._

**Answer:** $\hat{x}=[-1.342,-0.447,0.447,1.342]$, output $=[-2.683,-0.894,0.894,2.683]$. Only the 4 numbers of this one example were used — no other example entered the calculation, which is exactly why the answer is the same at any batch size.

</details>

**Problem 2.** Ablation / contrast: take a batch of two examples and switch the normalization axis from "across features" (LayerNorm) to "down the batch" (BatchNorm). What changes?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Use the batch [[2,4,4,6],[1,3,5,7]]. LayerNorm normalizes EACH ROW using that row's own mean/std. — _Per-example statistics — row 1 uses $\mu=4,\sigma=\sqrt2$; row 2 uses $\mu=4,\sigma=\sqrt5$._
- Now switch to averaging down each COLUMN (BatchNorm): feature 0 over both examples is [2,1], mean 1.5; etc. — _Per-feature statistics shared across the two examples._
- Observe: under LN, removing or adding a row does not change another row's output; under BN it does. — _LN's statistics are independent of the rest of the batch; BN's are not._

**Answer:** Switching the axis changes which numbers are pooled. LayerNorm pools across the 4 features within each example (output of a row is unchanged if you drop the other row); BatchNorm pools each feature across the 2 examples (output of a row depends on the other row). This is why LN is batch-size independent and BN is not — the core contrast with $dl\text{-}batchnorm$.

</details>

**Problem 3.** Why does LayerNorm need no train/eval distinction while BatchNorm does?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall BN at test time may see one example, so it cannot compute a batch mean — it falls back to running statistics gathered during training. — _BN's statistics require a batch._
- Note LN's statistics come from a single example's own features, available identically at train and test. — _Nothing depends on the batch, so nothing needs to be stored or swapped._

**Answer:** LayerNorm computes $\mu,\sigma$ from one example's features, which is exactly the same operation whether you are training on a big batch or serving a single request. So it performs "exactly the same computation at training and test times" (abstract) and needs no running averages and no eval-mode branch — unlike BatchNorm.

</details>